In [1]:
import os
import numpy as np
import psycopg
import pandas as pd
import mlflow
from catboost import CatBoostClassifier
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import psycopg
from sklearn.model_selection import train_test_split
import optuna
from optuna.integration import MLflowCallback
import mlflow
from collections import defaultdict
from mlflow.utils.mlflow_tags import MLFLOW_PARENT_RUN_ID
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
)
from numpy import median, array

load_dotenv("/home/mle-user/mle_projects/mle-mlflow/.env")

TABLE_NAME = "users_churn" # таблица с данными в postgres 

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ['AWS_ACCESS_KEY_ID'] = os.getenv('S3_ACCESS_KEY')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("S3_SECRET_KEY")

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

EXPERIMENT_NAME = "grid_random_search"
RUN_NAME = "model_bayesian_search" 
REGISTRY_MODEL_NAME = "cat_boost_search_v1"
FS_ASSETS = "fs_assets"

STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME = "churn_model"

/home/mle-user/.local/lib/python3.10/site-packages/pydantic/_internal/_fields.py:151: UserWarning: Field "model_server_url" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/home/mle-user/.local/lib/python3.10/site-packages/pydantic/_internal/_config.py:322: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)
/home/mle-user/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
active = mlflow.active_run()
while active:
    mlflow.end_run()
    active = mlflow.active_run()

try:
    optuna.delete_study(study_name=STUDY_NAME, storage=STUDY_DB_NAME)
except:
    pass

In [3]:
from mlflow.tracking.fluent import _active_run_stack

while len(_active_run_stack) > 0:
    mlflow.end_run()

In [4]:
df = pd.read_csv(r"~/mle_projects/mle-dvc/data/initial_data.csv")

features = ["monthly_charges", "total_charges", "senior_citizen"]
target = "target"

split_column = "begin_date"
stratify_column = "target"
test_size = 0.2

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(df[features], df[target], test_size=test_size, shuffle=False)

In [5]:
from numpy import median, array
from collections import defaultdict
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, precision_score,
    recall_score, f1_score, log_loss,
)
import optuna
from optuna.integration import MLflowCallback
import mlflow
from mlflow.utils.mlflow_tags import MLFLOW_PARENT_RUN_ID


# === 0. чистим висящие runs и старое study ===
active = mlflow.active_run()
while active:
    mlflow.end_run()
    active = mlflow.active_run()

try:
    optuna.delete_study(study_name=STUDY_NAME, storage=STUDY_DB_NAME)
except Exception:
    pass


# === 1. objective ===
def objective(trial: optuna.Trial) -> float:
    param = {
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
        "depth": trial.suggest_int("depth", 1, 12),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.1, 5),
        "random_strength": trial.suggest_float("random_strength", 0.1, 5),
        "loss_function": "Logloss",
        "task_type": "CPU",
        "random_seed": 0,
        "iterations": 300,
        "verbose": False,
    }

    model = CatBoostClassifier(**param)
    skf = StratifiedKFold(n_splits=2)
    metrics = defaultdict(list)

    for i, (train_index, val_index) in enumerate(skf.split(X_train, y_train)):
        train_x = X_train.iloc[train_index]
        val_x = X_train.iloc[val_index]
        train_y = y_train.iloc[train_index]
        val_y = y_train.iloc[val_index]

        model.fit(train_x, train_y, verbose=0)
        prediction = model.predict(val_x)
        probas = model.predict_proba(val_x)[:, 1]

        cm = confusion_matrix(val_y, prediction, normalize='all').ravel()
        err1 = cm[1]
        err2 = cm[2]
        auc = roc_auc_score(val_y, probas)
        precision = precision_score(val_y, prediction)
        recall = recall_score(val_y, prediction)
        f1 = f1_score(val_y, prediction)
        logloss = log_loss(val_y, prediction)

        metrics["err1"].append(err1)
        metrics["err2"].append(err2)
        metrics["auc"].append(auc)
        metrics["precision"].append(precision)
        metrics["recall"].append(recall)
        metrics["f1"].append(f1)
        metrics["logloss"].append(logloss)

    err_1 = median(array(metrics['err1']))
    err_2 = median(array(metrics['err2']))
    auc = median(array(metrics['auc']))
    precision = median(array(metrics['precision']))
    recall = median(array(metrics['recall']))
    f1 = median(array(metrics['f1']))
    logloss = median(array(metrics['logloss']))
    return auc


# === 2. эксперимент ===
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if not experiment:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id


# === 3. родительский run открывается и сразу закрывается ===
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id


# === 4. оптимизация (активного run нет — callback создаёт child runs) ===
mlflc = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="AUC",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id}})

study = optuna.create_study(
    direction='maximize',
    study_name=STUDY_NAME,
    sampler=optuna.samplers.TPESampler(),
    storage=STUDY_DB_NAME,
    load_if_exists=False)

study.optimize(objective, n_trials=10, callbacks=[mlflc])
best_params = study.best_params

print(f"Number of finished trials: {len(study.trials)}")
print(f"Best params: {best_params}")


# === 5. финальная модель в родительский run ===
best_params.update({
    "loss_function": "Logloss",
    "task_type": "CPU",
    "random_seed": 0,
    "iterations": 300,
    "verbose": False,
})

final_model = CatBoostClassifier(**best_params)
final_model.fit(X_train, y_train)
final_model.save_model("model.cb")

with mlflow.start_run(run_id=run_id):
    mlflow.catboost.log_model(cb_model=final_model, artifact_path="cv")
    mlflow.log_artifact("model.cb", artifact_path="cv")

/tmp/ipykernel_42191/2869723484.py:96: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflc = MLflowCallback(
[I 2026-06-21 09:16:52,632] A new study created in RDB with name: churn_model
/home/mle-user/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
[I 2026-06-21 09:16:54,537] Trial 0 finished with value: 0.7640667721085546 and parameters: {'learning_rate': 0.0027859020319642705, 'depth': 10, 'l2_leaf_reg': 0.15615859474147156, 'random_strength': 3.8584522395995138}. Best is trial 0 with value: 0.7640667721085546.
/home/mle-user/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to

Number of finished trials: 10
Best params: {'learning_rate': 0.03767280821721552, 'depth': 5, 'l2_leaf_reg': 2.3107200340831, 'random_strength': 3.3872406053524715}


In [6]:
run_id

'5cd1bebd64f840c297b065745ba126b3'